# FabriBuddy_Scan

Scans the tenant **read-only as the service principal** and stores what the **FabriBuddy**
app shows in `FabriBuddy_DB`. Run by the pipeline **Load_FabriBuddy_E2E**, which holds the
settings — change them there.

[Documentation](https://zan-tavcar.github.io/msfabric-fabribuddy/)

In [ ]:
# Set by the pipeline Load_FabriBuddy_E2E (edit its parameters, not these).
key_vault_url = ''
sp_tenant_id = ''
sp_client_id = ''
sp_secret_name = 'fabribuddy-sp-secret'
workspaces = ''
include_personal = False
tenant_name = ''
full_refresh = False
demo = False
sql_database_name = 'FabriBuddy_DB'

In [ ]:
# FabriBuddy_DB, before the FabriBuddy package is installed. As whoever runs this notebook.
import json, struct, subprocess, sys, time
import requests

FABRIC_API = "https://api.fabric.microsoft.com/v1"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]


def fabric(method, path, body=None, wait=True):
    """The Fabric REST API as you; waits for long-running operations."""
    headers = {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}"}
    r = requests.request(method, f"{FABRIC_API}{path}", headers=headers, json=body, timeout=120)
    if r.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed ({r.status_code}): {r.text[:400]}")
    if r.status_code == 202 and wait and r.headers.get("Location"):
        location, state = r.headers["Location"], None
        for _ in range(200):
            time.sleep(min(int(r.headers.get("Retry-After", 3) or 3), 10))
            r = requests.get(location, headers=headers, timeout=60)
            state = (r.json() if r.text else {}).get("status")
            if state in ("Succeeded", "Failed", "Undefined"):
                break
        if state != "Succeeded":
            raise RuntimeError(f"{method} {path} did not succeed: {r.text[:400]}")
        result = requests.get(location + "/result", headers=headers, timeout=60)
        return result.json() if result.status_code == 200 and result.text else {}
    return r.json() if r.text else {}


def find_sql_database(name):
    """(item id, server, database) of the SQL database called `name` here, or (None, "", "")."""
    for item in fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases").get("value", []):
        if item["displayName"].lower() == name.lower():
            props = fabric("GET", f"/workspaces/{WORKSPACE_ID}/sqlDatabases/{item['id']}").get("properties") or {}
            return item["id"], (props.get("serverFqdn") or "").split(",")[0], props.get("databaseName") or ""
    return None, "", ""


def sql_token():
    return notebookutils.credentials.getToken("https://database.windows.net/")


def sql_connect(server, database):
    import pyodbc

    raw = sql_token().encode("utf-16-le")
    return pyodbc.connect(
        f"Driver={{ODBC Driver 18 for SQL Server}};Server={server},1433;Database={database};"
        "Encrypt=yes;TrustServerCertificate=no",
        attrs_before={1256: struct.pack(f"<I{len(raw)}s", len(raw), raw)},
        timeout=30,
    )


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *args])

In [ ]:
# Install the FabriBuddy release that Deploy_FabriBuddy stored in the database.
_, SQL_SERVER, SQL_DATABASE = find_sql_database(sql_database_name)
if not SQL_SERVER:
    raise RuntimeError(f"No SQL database called {sql_database_name!r} here. Run Deploy_FabriBuddy first.")
with sql_connect(SQL_SERVER, SQL_DATABASE) as conn:
    row = conn.cursor().execute("SELECT name, version, data FROM dbo.fabribuddy_package").fetchone()
if row is None:
    raise RuntimeError("The FabriBuddy package isn't in the database. Run Deploy_FabriBuddy first.")
wheel = f"/tmp/{row[0]}"
with open(wheel, "wb") as f:
    f.write(bytes(row[2]))
pip_install(wheel)
print(f"FabriBuddy {row[1]}")

In [ ]:
# The service principal's secret, from Azure Key Vault (in memory only).
client_secret = ""
if not demo:
    missing = [n for n in ("key_vault_url", "sp_tenant_id", "sp_client_id") if not globals()[n]]
    if missing:
        raise ValueError(f"Set {', '.join(missing)} in the parameters of Load_FabriBuddy_E2E.")
    client_secret = notebookutils.credentials.getSecret(key_vault_url, sp_secret_name)

In [ ]:
# Scan and publish.
from fabribuddy_intenant import run

result = run(
    tenant_id=sp_tenant_id,
    client_id=sp_client_id,
    client_secret=client_secret,
    sql_server=SQL_SERVER,
    sql_database=SQL_DATABASE,
    sql_token=sql_token(),
    tenant_name=tenant_name,
    workspaces=[w for w in workspaces.split(",") if w.strip()],
    include_personal=include_personal,
    full=full_refresh,
    demo=demo,
)
print(json.dumps(result, indent=2, default=str)[:4000])